# Visual Classifier — Flexible Fine-Tuning Workbench

This notebook is a **flexible experimentation workbench** for fine-tuning
the `facebook/convnextv2-base-22k-384` model on Apple Silicon.

### How to use

1. **Run sections 0–2** once to set up the environment and load the Zitacron dataset.
2. **Edit the config block** in Section 3, then run the training cell.
3. **Evaluate** in Section 4.
4. **Repeat** — change the config and re-run Section 3 as many times as you want.

The dataset is `Zitacron/real-vs-ai-corpus` containing images from various generators.
The task is **binary classification**: `0 = Real`, `1 = AI-Generated`.

> ⚠️ **Memory:** Unified memory on Apple Silicon is used. Batch sizes are kept small to avoid OOM.

> ⚠️ **Large model checkpoints are .gitignored.** Only lightweight deltas or adapters should be committed.

---
## 0 · Shared Configuration
Global constants shared by all cells.  Edit once at the start.

In [1]:
# ── Base model ──
BASE_MODEL = "facebook/convnextv2-base-22k-384"

# ── Zitacron subset sizes ──
TOTAL_TRAIN = 10000
TOTAL_VAL   = 2500
TOTAL_TEST  = 2500
SEED        = 42

# ── Output paths ──
MODELS_DIR      = "outputs/models"
EVAL_OUTPUT_DIR  = "outputs"

---
## 1 · Environment Setup

In [2]:
import os, sys, torch

# ── GPU / Accelerator check ──
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("✅ MPS (Apple Silicon) available")
    USE_FP16 = False   # fp16 not fully supported on MPS
elif torch.cuda.is_available():
    print(f"✅ CUDA available – {torch.cuda.get_device_name(0)}")
    USE_FP16 = True
else:
    print("⚠️  No GPU – training will be slow on CPU")
    USE_FP16 = False

print(f"Mixed precision (fp16): {USE_FP16}")
print(f"PyTorch version: {torch.__version__}")

✅ MPS (Apple Silicon) available
Mixed precision (fp16): False
PyTorch version: 2.11.0


### Authentication

Zitacron/real-vs-ai-corpus may require a Hugging Face token.

**Do NOT hardcode your token.** Use one of these safe methods:
- Set the `HF_TOKEN` environment variable before launching the notebook.
- Or run `huggingface_hub.login()` / `notebook_login()` interactively.

In [3]:
HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN is None:
    try:
        from huggingface_hub import notebook_login
        notebook_login()          # interactive widget / prompt
        HF_TOKEN = True           # login() stores the token globally
    except Exception:
        print("⚠️  No HF token found. Public datasets will still work,"
              " but gated datasets may fail.")
        HF_TOKEN = None
else:
    print("✅ HF_TOKEN loaded from environment.")

### Imports

In [4]:
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoImageProcessor, AutoModelForImageClassification

# Our helper module (lives next to this notebook)
from custom_dataset_builder import (
    get_device,
    load_model_from,
    build_custom_dataset,
    run_training_stage,
    evaluate_model,
)
# Weight-delta helpers from the existing visual_classifier module
from visual_classifier import save_weight_delta

DEVICE = get_device()
print(f"Device: {DEVICE}")

Device: mps


---
## 2 · Load Zitacron Dataset

Run this cell once to load and build the balanced Zitacron subset.

In [5]:
zitacron_ds = build_custom_dataset(
    total_train=TOTAL_TRAIN,
    total_val=TOTAL_VAL,
    total_test=TOTAL_TEST,
    seed=SEED,
    hf_token=HF_TOKEN if isinstance(HF_TOKEN, str) else None,
)
zitacron_ds

📡 Pulling up to 1500 images from ronantakizawa/webui...
   ... streamed 500 rows ...
   ... streamed 1000 rows ...
   ✅ Collected 1500 images.
📡 Pulling up to 1500 images from derek-thomas/ScienceQA...
   ... streamed 500 rows ...
   ... streamed 1000 rows ...
   ... streamed 1500 rows ...
   ... streamed 2000 rows ...
   ... streamed 2500 rows ...
   ... streamed 3000 rows ...
   ✅ Collected 1500 images.
📡 Pulling up to 1500 images from MBZUAI/OpenEarthAgent...


Resolving data files:   0%|          | 0/11657 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/403 [00:00<?, ?it/s]

   ... streamed 500 rows ...
   ... streamed 1000 rows ...
   ✅ Collected 1500 images.
📡 Pulling up to 1500 images from EPFL-ECEO/coralscapes...
   ... streamed 500 rows ...
   ... streamed 1000 rows ...
   ✅ Collected 1500 images.
📡 Pulling up to 1500 images from opendatalab/OmniDocBench...


README.md: 0.00B [00:00, ?B/s]

...

   ... streamed 500 rows ...
   ... streamed 1000 rows ...
   ✅ Collected 1500 images.
📡 Pulling up to 1500 images from Sigurdur/isl-finepdfs-images...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

KeyboardInterrupt: 

---
## 3 · 🔧 Training Run

**Edit the config block below**, then run the cell.  
Re-run this cell as many times as you want with different settings.

#### `START_FROM` options
| Value | Meaning |
|-------|---------|
| `"base"` | Fresh start from the original facebook/convnextv2-base-22k-384 model |
| `"outputs/models/run_01_zitacron"` | Continue from a previously saved checkpoint |
| `"current"` | Continue from whatever model was trained in the last run (still in memory) |


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  🔧  TRAINING RUN CONFIG — edit this block, then run the cell
# ═══════════════════════════════════════════════════════════════
RUN_NAME         = "run_01_zitacron"
START_FROM       = "base"
TRAIN_DATASET    = "zitacron"
EPOCHS           = 3
BATCH_SIZE       = 8
LEARNING_RATE    = 2e-5
AUGMENT          = True
WEIGHT_DECAY     = 0.05
EARLY_STOPPING   = 2

In [ ]:
# ── Resolve starting model ──
if START_FROM == "current":
    assert 'current_model' in dir() and current_model is not None, \
        "No current_model in memory. Use 'base' or a saved checkpoint path."
    model = current_model
    print(f"📦  Continuing from current in-memory model")
else:
    model, processor = load_model_from(source=START_FROM, device=DEVICE)

# ── Resolve datasets ──
_datasets = {
    "zitacron":    zitacron_ds    if 'zitacron_ds'    in dir() else None,
}

train_data = _datasets[TRAIN_DATASET]
assert train_data is not None, f"Dataset '{TRAIN_DATASET}' not loaded. Run the loader cell first."

# ── Output directory ──
run_output_dir = os.path.join(MODELS_DIR, RUN_NAME)

# ── Print summary ──
print(f"\n{'─'*50}")
print(f"  Run:           {RUN_NAME}")
print(f"  Start from:    {START_FROM}")
print(f"  Train on:      {TRAIN_DATASET} ({len(train_data['train'])} train samples)")
print(f"  Epochs:        {EPOCHS}")
print(f"  Batch size:    {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Augment:       {AUGMENT}")
print(f"  Output:        {run_output_dir}")
print(f"{'─'*50}\n")

# ── Train ──
current_model, current_trainer = run_training_stage(
    model=model,
    processor=processor,
    train_ds=train_data["train"],
    eval_ds=train_data["validation"],
    output_dir=run_output_dir,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    stage_name=RUN_NAME,
    fp16=USE_FP16,
    augment=AUGMENT,
    weight_decay=WEIGHT_DECAY,
    early_stopping_patience=EARLY_STOPPING,
)

print(f"\n✅  Training complete.  Model stored in `current_model`.")
print(f"    Saved to: {run_output_dir}")
print(f"    To continue training from this model, set START_FROM = 'current' or '{run_output_dir}'")

📦  Loading model from: facebook/convnextv2-base-22k-384


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

[transformers] ConvNextV2ForImageClassification LOAD REPORT from: facebook/convnextv2-base-22k-384
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])            
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 1024]) vs model:torch.Size([2, 1024])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


✅  Model loaded  –  87,694,850 parameters  (device: mps)

──────────────────────────────────────────────────
  Run:           run_01_zitacron
  Start from:    base


KeyError: 'train'

---
## 4 · 📊 Evaluate Current Model

Evaluates `current_model` on Zitacron test set.
Edit the config, then run.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  📊  EVALUATION CONFIG — edit this block, then run the cell
# ═══════════════════════════════════════════════════════════════
EVAL_ON     = ["zitacron"]
EVAL_PREFIX = RUN_NAME
EVAL_BATCH  = 8

In [ ]:
assert 'current_model' in dir() and current_model is not None, \
    "No current_model to evaluate. Run a training cell first."

_test_sets = {
    "zitacron":    zitacron_ds["test"]    if 'zitacron_ds'    in dir() else None,
}

eval_results = {}
for ds_name in EVAL_ON:
    test_ds = _test_sets.get(ds_name)
    if test_ds is None:
        print(f"⚠️  Skipping {ds_name} — dataset not loaded")
        continue
    print(f"\n{'='*50}")
    print(f"  Evaluating on: {ds_name} ({len(test_ds)} samples)")
    print(f"{'='*50}")

    prefix = f"{EVAL_PREFIX}_on_ds_{ds_name}"
    metrics = evaluate_model(
        model=current_model,
        processor=processor,
        test_ds=test_ds,
        output_prefix=prefix,
        output_dir=EVAL_OUTPUT_DIR,
        batch_size=EVAL_BATCH,
        fp16=USE_FP16,
    )
    eval_results[ds_name] = metrics

# ── Quick comparison table ──
if eval_results:
    print(f"\n{'='*60}")
    print(f"  📋  Summary for: {EVAL_PREFIX}")
    print(f"{'='*60}")
    print(f"  {'Dataset':<15} {'Accuracy':>10} {'F1':>10} {'Precision':>10} {'Recall':>10}")
    print(f"  {'─'*55}")
    for name, m in eval_results.items():
        print(f"  {name:<15} {m['accuracy']:>10.4f} {m['f1']:>10.4f} {m['precision']:>10.4f} {m['recall']:>10.4f}")

---
## 5 · 💾 Save Weight Delta

Save only the **weight differences** from the base model.  
The resulting file is typically ~74 MB (int8 quantised) and stays under GitHub's 100 MB limit.

> This is **full fine-tuning**, not LoRA/PEFT. The delta is not a
> lightweight adapter — it encodes changes across all parameters.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  💾  DELTA CONFIG — edit this block, then run the cell
# ═══════════════════════════════════════════════════════════════
DELTA_OUTPUT_PATH = f"./fine_tuned_model_delta/{RUN_NAME}_weight_delta.pt"

In [ ]:
# Reload module to pick up any fixes
import importlib, visual_classifier
importlib.reload(visual_classifier)
from visual_classifier import save_weight_delta

assert 'current_model' in dir() and current_model is not None, \
    "No current_model to save. Run a training cell first."

delta_path, delta_mb = save_weight_delta(
    fine_tuned_model=current_model,
    base_model_name=BASE_MODEL,
    output_path=DELTA_OUTPUT_PATH,
)
print(f"\n✅ Delta saved to '{delta_path}' ({delta_mb:.2f} MB)")

---
## 6 · 🔍 Compare Models on Sample Images

Load any combination of saved models and compare predictions side-by-side.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  🔍  COMPARISON CONFIG — edit this block, then run the cell
# ═══════════════════════════════════════════════════════════════
# Each entry: (display_name, model_name_or_path, delta_path_or_None)
MODELS_TO_COMPARE = [
    ("Base",                        "facebook/convnextv2-base-22k-384",    None),
    ("Zitacron Fine-tuned",         "outputs/models/run_01_zitacron",              None),
    # ("Delta",      "facebook/convnextv2-base-22k-384", "fine_tuned_model_delta/run_01_zitacron_weight_delta.pt"),
]

SAMPLE_DIR = "../../data/sample_images"

In [ ]:
from visual_classifier import VisualClassifier
from PIL import Image

# ── Load all comparison models ──
loaded_models = []
for display_name, model_path, delta in MODELS_TO_COMPARE:
    print(f"Loading: {display_name} ...")
    vc = VisualClassifier(
        model_name_or_path=model_path,
        delta_path=delta,
    )
    loaded_models.append((display_name, vc))

print(f"\n✅  Loaded {len(loaded_models)} models for comparison\n")

# ── Run predictions on sample images ──
for fname in sorted(os.listdir(SAMPLE_DIR)):
    if fname.startswith("."):
        continue
    img = Image.open(os.path.join(SAMPLE_DIR, fname))
    expected = "Real" if fname.startswith("real") else "AI Generated"

    print(f"\n{fname}")
    print(f"  Expected: {expected}")
    for display_name, vc in loaded_models:
        result = vc.predict(img)
        marker = "✅" if result['prediction'] == expected else "❌"
        print(f"  {marker} {display_name:>15}:  {result['prediction']} ({result['confidence']:.2%})")

---
## 📝 Quick Reference — Example Workflows

### Workflow A: Fine-tune from base model
```
Run 1:  RUN_NAME="run_01_zitacron",    START_FROM="base"
```

### Workflow B: Continue training from previous run
```
Run 1:  RUN_NAME="run_02_continued",   START_FROM="outputs/models/run_01_zitacron"
```